# Retroviral Wall Challenge — Visualisations

Self-contained notebook. Reproduces v14 OOF predictions inside LOFO loop,
then generates all 10 figures. Figures saved to `figures/` at 150 dpi.

In [ ]:
import warnings, json
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import average_precision_score
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import xgboost as xgb
from catboost import CatBoostRegressor

DATA_DIR = Path('C:/Users/yatin/Downloads/mandrake_data_25_03/mandrake_data')
FIG_DIR  = DATA_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)
SEED = 42
np.random.seed(SEED)

PALETTE = sns.color_palette('Set2', 7)
DPI = 150
print('Setup done. figures/ ready at:', FIG_DIR)

## Load data

In [ ]:
rt_seq  = pd.read_csv(DATA_DIR / 'rt_sequences.csv')
feat_df = pd.read_csv(DATA_DIR / 'handcrafted_features.csv')
df = feat_df.merge(
    rt_seq[['rt_name','active','pe_efficiency_pct','rt_family','protein_length_aa']],
    on='rt_name'
)
df['pe_efficiency_pct'] = df['pe_efficiency_pct'].fillna(0.0)
emb_data  = np.load(str(DATA_DIR / 'esm2_embeddings.npz'), allow_pickle=True)
esm2_dict = dict(zip(emb_data['names'], emb_data['embeddings']))
all_embs  = np.vstack([esm2_dict[n] for n in df['rt_name']])

# Engineered features (same as run_v14_final.py)
df['TM_ratio_MMLVPE_telo'] = df['foldseek_TM_MMLVPE'] / (df['foldseek_TM_Telomerase'] + 1e-6)
df['is_broken_enzyme']     = df['triad_best_rmsd'].isna().astype(int)
df['length_to_TM_ratio']   = df['protein_length_aa'] / (df['foldseek_best_TM'] + 1e-5)
df['is_no_thumb']   = df['thumb_fident'].isna().astype(int)
df['is_no_hairpin'] = df['n_hairpins_found'].isna().astype(int)

FAMILIES = sorted(df['rt_family'].unique())
FAM_COLORS = {f: PALETTE[i] for i, f in enumerate(FAMILIES)}
print(f'Loaded {len(df)} RTs across {len(FAMILIES)} families')
print(f'Active: {df["active"].sum()} / {len(df)}')

## Reproduce v14 OOF predictions (LOFO loop)

In [ ]:
FOLDSEEK = ['foldseek_best_TM','foldseek_best_fident','foldseek_best_LDDT',
            'foldseek_TM_HIV1','foldseek_TM_MMLV','foldseek_TM_MMLVPE',
            'foldseek_TM_Telomerase','foldseek_TM_LTRRetrotransposon',
            'foldseek_TM_Group2Intron','foldseek_TM_Retron']
BASE   = ['triad_best_rmsd','D1_D2_dist','pocket_hydrophobic_per_res',
          'camsol_score','perplexity','instability_index',
          'native_net_charge','protein_length_aa','TM_ratio_MMLVPE_telo']
THERMO = ['t55_raw','t60_raw','t65_raw']
ENG    = ['is_broken_enzyme','length_to_TM_ratio','is_no_thumb','is_no_hairpin']
PCA_N  = 5
PCA_COLS = [f'pca_{i}' for i in range(PCA_N)]
for c in PCA_COLS:
    df[c] = 0.0

CLF_FEATURES = FOLDSEEK + BASE + ['hbonds_per_res','n_strands_total'] + ENG + PCA_COLS
REG_FEATURES = FOLDSEEK + BASE + ['thumb_fident','n_hairpins_found'] + THERMO + ENG + ['isoelectric_point','n_salt_bridges'] + PCA_COLS

y_bin    = df['active'].values
y_eff    = df['pe_efficiency_pct'].values
families = df['rt_family'].values
clf_oof  = np.zeros(len(df))
reg_xgb  = np.zeros(len(df))
reg_cb   = np.zeros(len(df))

def _fill(X): return np.where(np.isnan(X), -999.0, X)
def _lr():
    return Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler()),
        ('m',   LogisticRegression(C=0.1, penalty='l2', class_weight='balanced',
                                    max_iter=2000, random_state=SEED))
    ])

for fam in sorted(set(families)):
    tr = np.where(families != fam)[0]
    te = np.where(families == fam)[0]
    pf = PCA(n_components=PCA_N, random_state=SEED)
    pt = pf.fit_transform(all_embs[tr])
    pe = pf.transform(all_embs[te])
    for i in range(PCA_N):
        df.iloc[tr, df.columns.get_loc(f'pca_{i}')] = pt[:, i]
        df.iloc[te, df.columns.get_loc(f'pca_{i}')] = pe[:, i]
    Xct = df.iloc[tr][CLF_FEATURES].values.astype(float)
    Xce = df.iloc[te][CLF_FEATURES].values.astype(float)
    Xrt = _fill(df.iloc[tr][REG_FEATURES].values.astype(float))
    Xre = _fill(df.iloc[te][REG_FEATURES].values.astype(float))
    yb = y_bin[tr]; ye = y_eff[tr]; sw = np.sqrt(ye + 0.01)
    lr = _lr()
    lr.fit(Xct, yb)
    clf_oof[te] = lr.predict_proba(Xce)[:, 1]
    m = xgb.XGBRegressor(n_estimators=200, max_depth=2, learning_rate=0.03,
                          subsample=0.8, colsample_bytree=0.8,
                          random_state=SEED, verbosity=0)
    m.fit(Xrt, ye, sample_weight=sw)
    reg_xgb[te] = np.clip(m.predict(Xre), 0, None)
    m = CatBoostRegressor(iterations=200, depth=2, learning_rate=0.03,
                           verbose=False, random_seed=SEED)
    m.fit(Xrt, ye, sample_weight=sw)
    reg_cb[te] = np.clip(m.predict(Xre), 0, None)
    print(f'  {fam:<25} n={len(te)} active={int(y_bin[te].sum())}')

reg_oof = (reg_xgb + reg_cb) / 2.0

# Fine beta sweep to find optimal
def weighted_spearman(pred, true_eff, weights):
    pr = np.argsort(np.argsort(pred)).astype(float)
    tr = np.argsort(np.argsort(true_eff)).astype(float)
    w  = weights / weights.sum()
    mp = np.dot(w, pr); mt = np.dot(w, tr)
    cov = np.sum(w * (pr - mp) * (tr - mt))
    sp = np.sqrt(np.sum(w * (pr - mp)**2))
    st = np.sqrt(np.sum(w * (tr - mt)**2))
    if sp < 1e-12 or st < 1e-12: return 0.0
    return max(cov / (sp * st), 0.0)

def compute_cls(y_true, y_score, pe_eff):
    pr = average_precision_score(np.array(y_true), y_score)
    ws = weighted_spearman(y_score, np.array(pe_eff), np.array(pe_eff) + 0.01)
    return (2 * pr * ws / (pr + ws) if pr > 0 and ws > 0 else 0.0), pr, ws

beta_vals, cls_vals, pr_vals, ws_vals = [], [], [], []
best_cls, best_beta = -1.0, 0.66
for beta in np.arange(0.0, 1.01, 0.01):
    f = beta * (rankdata(clf_oof) / len(df)) + (1 - beta) * (rankdata(reg_oof) / len(df))
    c, p, w = compute_cls(y_bin, f, y_eff)
    beta_vals.append(beta); cls_vals.append(c); pr_vals.append(p); ws_vals.append(w)
    if c > best_cls:
        best_cls, best_beta = c, float(beta)

oof = best_beta * (rankdata(clf_oof) / len(df)) + (1 - best_beta) * (rankdata(reg_oof) / len(df))
cls_final, pr_final, ws_final = compute_cls(y_bin, oof, y_eff)
print(f'\nV14 OOF: CLS={cls_final:.4f}  PR-AUC={pr_final:.4f}  WSpearman={ws_final:.4f}  beta={best_beta:.2f}')

## Fig 01 — Family distribution

In [ ]:
# fig_01_family_distribution.png — stacked bar: n_active and n_inactive per family,
# sorted by n_active descending. Retroviral highlighted distinctly.

fam_counts = df.groupby('rt_family').agg(
    n_active=('active','sum'),
    n_inactive=('active', lambda x: (x==0).sum())
).sort_values('n_active', ascending=False).reset_index()

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(fam_counts))
retro_idx = fam_counts.index[fam_counts['rt_family'] == 'Retroviral'].tolist()

bars_act = ax.bar(x, fam_counts['n_active'],   color='#4CAF50', label='Active',   edgecolor='k', linewidth=0.6)
bars_ina = ax.bar(x, fam_counts['n_inactive'], color='#EF5350', label='Inactive', edgecolor='k', linewidth=0.6,
                  bottom=fam_counts['n_active'])

# Highlight Retroviral with a distinct border
for i in retro_idx:
    ax.bar(i, fam_counts.loc[i,'n_active'] + fam_counts.loc[i,'n_inactive'],
           color='none', edgecolor='navy', linewidth=2.5)

ax.set_xticks(x)
ax.set_xticklabels(fam_counts['rt_family'], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Number of RTs')
ax.set_title('RT Family Distribution (sorted by active count)')
ax.legend()
ax.text(retro_idx[0], fam_counts.loc[retro_idx[0],'n_active'] + fam_counts.loc[retro_idx[0],'n_inactive'] + 0.2,
        'Retroviral\n(highlighted)', ha='center', fontsize=8, color='navy')
# Caption: Retroviral family has the most active RTs (12/18); CRISPR-associated and Other have zero active members.
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_01_family_distribution.png', dpi=DPI)
plt.close()
print('fig_01 saved')

## Fig 02 — Efficiency distribution by family

In [ ]:
# fig_02_efficiency_distribution.png — strip/swarm plot of pe_efficiency_pct by family (active RTs only).
# MMLV-RT labelled explicitly. Annotate the efficiency gap between Retroviral and non-Retroviral actives.

act_df = df[df['active'] == 1].copy()
fam_order = act_df.groupby('rt_family')['pe_efficiency_pct'].max().sort_values(ascending=False).index.tolist()

fig, ax = plt.subplots(figsize=(10, 5))
sns.stripplot(data=act_df, x='rt_family', y='pe_efficiency_pct', order=fam_order,
              palette={f: FAM_COLORS[f] for f in fam_order}, size=8, jitter=0.15,
              edgecolor='k', linewidth=0.5, ax=ax)

# Label MMLV-RT
mmlv_row = act_df[act_df['rt_name'] == 'MMLV-RT']
if not mmlv_row.empty:
    xi = fam_order.index('Retroviral')
    ax.annotate('MMLV-RT', xy=(xi, mmlv_row['pe_efficiency_pct'].values[0]),
                xytext=(xi + 0.5, mmlv_row['pe_efficiency_pct'].values[0] + 1.5),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.2),
                fontsize=8, ha='left')

# Annotate efficiency gap
retro_max  = act_df[act_df['rt_family'] == 'Retroviral']['pe_efficiency_pct'].max()
nonretro_max = act_df[act_df['rt_family'] != 'Retroviral']['pe_efficiency_pct'].max()
ax.axhline(retro_max,    color='green', lw=0.8, ls='--', alpha=0.6)
ax.axhline(nonretro_max, color='orange', lw=0.8, ls='--', alpha=0.6)
ax.text(len(fam_order) - 0.5, retro_max + 0.3, f'Retroviral max: {retro_max:.1f}%', fontsize=7, color='green', ha='right')
ax.text(len(fam_order) - 0.5, nonretro_max + 0.3, f'Non-Retroviral max: {nonretro_max:.1f}%', fontsize=7, color='orange', ha='right')

ax.set_xlabel('Family')
ax.set_ylabel('PE Efficiency (%)')
ax.set_title('Prime Editing Efficiency by Family (active RTs only)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right', fontsize=9)
# Caption: Retroviral RTs dominate the high-efficiency regime; Er-RT (Group II Intron) at 12.5% is the highest non-Retroviral active.
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_02_efficiency_distribution.png', dpi=DPI)
plt.close()
print('fig_02 saved')

## Fig 03 — Feature correlations

In [ ]:
# fig_03_feature_correlations.png — horizontal bar chart, top 20 features by |Spearman rho|
# with pe_efficiency (actives only). Colour bars by sign.
# Secondary column shows correlation with active/inactive label.

num_cols = [c for c in df.columns if c not in
            ['rt_name','active','pe_efficiency_pct','rt_family','sequence'] +
            [f'pca_{i}' for i in range(5)]]

act_df2 = df[df['active'] == 1]
corr_rows = []
for c in num_cols:
    try:
        v = pd.to_numeric(df[c], errors='coerce')
        va = pd.to_numeric(act_df2[c], errors='coerce')
        if v.notna().sum() < 20 or va.notna().sum() < 10: continue
        r_eff, _ = spearmanr(va.dropna(), act_df2.loc[va.dropna().index, 'pe_efficiency_pct'])
        r_act, _ = spearmanr(v.dropna(), df.loc[v.dropna().index, 'active'])
        if not np.isnan(r_eff) and not np.isnan(r_act):
            corr_rows.append({'feature': c, 'rho_eff': r_eff, 'rho_act': r_act})
    except:
        pass

corr_df = pd.DataFrame(corr_rows).sort_values('rho_eff', key=abs, ascending=False).head(20)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 7), sharey=True)
colors_eff = ['#4CAF50' if r > 0 else '#EF5350' for r in corr_df['rho_eff']]
colors_act = ['#4CAF50' if r > 0 else '#EF5350' for r in corr_df['rho_act']]

ax1.barh(corr_df['feature'], corr_df['rho_eff'], color=colors_eff, edgecolor='k', linewidth=0.4)
ax1.axvline(0, color='k', lw=0.8)
ax1.set_xlabel('Spearman rho with PE efficiency (actives only)')
ax1.set_title('Correlation with Efficiency')

ax2.barh(corr_df['feature'], corr_df['rho_act'], color=colors_act, edgecolor='k', linewidth=0.4)
ax2.axvline(0, color='k', lw=0.8)
ax2.set_xlabel('Spearman rho with active label (all 57)')
ax2.set_title('Correlation with Active/Inactive')

fig.suptitle('Top 20 Features by |rho| with PE Efficiency', fontsize=12)
# Caption: TM_ratio_MMLVPE_telo and foldseek scores dominate both efficiency and activity correlations.
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_03_feature_correlations.png', dpi=DPI)
plt.close()
print('fig_03 saved')

## Fig 04 — FoldSeek landscape

In [ ]:
# fig_04_foldseek_landscape.png — scatter: foldseek_TM_MMLV vs foldseek_TM_Telomerase,
# coloured by family, shaped by active/inactive. Diagonal = equal TM.

fig, ax = plt.subplots(figsize=(9, 7))
markers = {1: 'o', 0: 'X'}

for fam in FAMILIES:
    for act_val, mk in markers.items():
        sub = df[(df['rt_family'] == fam) & (df['active'] == act_val)]
        if len(sub) == 0: continue
        ax.scatter(sub['foldseek_TM_MMLV'], sub['foldseek_TM_Telomerase'],
                   color=FAM_COLORS[fam], marker=mk, s=70 if act_val == 1 else 50,
                   edgecolors='k', linewidths=0.5, alpha=0.85,
                   label=f'{fam} ({"active" if act_val else "inactive"})' if mk == 'o' else '_nolegend_')

# Diagonal line (equal TM to both refs)
lims = [0, 1]
ax.plot(lims, lims, 'k--', lw=0.8, alpha=0.5, label='Equal TM line')

# Annotate key RTs
for rt_name, label in [('MMLV-RT','MMLV-RT'), ('Tf1-RT','Tf1-RT')]:
    row = df[df['rt_name'] == rt_name]
    if not row.empty:
        ax.annotate(label,
                    xy=(row['foldseek_TM_MMLV'].values[0], row['foldseek_TM_Telomerase'].values[0]),
                    xytext=(row['foldseek_TM_MMLV'].values[0] + 0.02,
                            row['foldseek_TM_Telomerase'].values[0] + 0.02),
                    fontsize=8, arrowprops=dict(arrowstyle='->', lw=0.8))

ax.set_xlabel('FoldSeek TM-score to MMLV')
ax.set_ylabel('FoldSeek TM-score to Telomerase')
ax.set_title('FoldSeek Structural Landscape: MMLV vs Telomerase TM-scores')
# Compact legend (families only)
handles = [mpatches.Patch(color=FAM_COLORS[f], label=f) for f in FAMILIES]
handles += [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='grey', ms=8, label='Active'),
            plt.Line2D([0],[0], marker='X', color='w', markerfacecolor='grey', ms=8, label='Inactive')]
ax.legend(handles=handles, fontsize=7, loc='upper left', ncol=2)
# Caption: Retroviral RTs cluster toward high MMLV / low Telomerase TM-scores; the ratio TM_ratio_MMLVPE_telo captures this separation.
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_04_foldseek_landscape.png', dpi=DPI)
plt.close()
print('fig_04 saved')

## Fig 05 — ESM-2 PCA

In [ ]:
# fig_05_esm2_pca.png — PCA (PC1 vs PC2) from ESM-2 embeddings,
# coloured by family, shaped by active/inactive. Annotate MMLV-RT, Tf1-RT, Er-RT, Gs-RT.

pca_global = PCA(n_components=2, random_state=SEED)
pc = pca_global.fit_transform(all_embs)

fig, ax = plt.subplots(figsize=(9, 7))
markers = {1: 'o', 0: 'X'}
for fam in FAMILIES:
    idx = df[df['rt_family'] == fam].index
    for act_val, mk in markers.items():
        sub_idx = df[(df['rt_family'] == fam) & (df['active'] == act_val)].index
        if len(sub_idx) == 0: continue
        pos = [df.index.get_loc(i) for i in sub_idx]
        ax.scatter(pc[pos, 0], pc[pos, 1],
                   color=FAM_COLORS[fam], marker=mk, s=70 if act_val else 50,
                   edgecolors='k', linewidths=0.5, alpha=0.85)

for rt_name in ['MMLV-RT', 'Tf1-RT', 'Er-RT', 'Gs-RT']:
    row = df[df['rt_name'] == rt_name]
    if row.empty: continue
    pos = df.index.get_loc(row.index[0])
    ax.annotate(rt_name, xy=(pc[pos, 0], pc[pos, 1]),
                xytext=(pc[pos, 0] + 0.5, pc[pos, 1] + 0.3),
                fontsize=8, arrowprops=dict(arrowstyle='->', lw=0.8))

ax.set_xlabel(f'PC1 ({pca_global.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca_global.explained_variance_ratio_[1]*100:.1f}% var)')
ax.set_title('ESM-2 Embeddings: PCA (global, for visualisation only)')
handles = [mpatches.Patch(color=FAM_COLORS[f], label=f) for f in FAMILIES]
handles += [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='grey', ms=8, label='Active'),
            plt.Line2D([0],[0], marker='X', color='w', markerfacecolor='grey', ms=8, label='Inactive')]
ax.legend(handles=handles, fontsize=7, loc='best', ncol=2)
# Caption: ESM-2 PCA separates families along PC1; active/inactive do not cluster perfectly, motivating the handcrafted feature approach.
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_05_esm2_pca.png', dpi=DPI)
plt.close()
print('fig_05 saved')

## Fig 06 — Version comparison

In [ ]:
# fig_06_version_comparison.png — grouped bar chart comparing CLS, PR-AUC, WSpearman
# across versions. Show delta at each step.

versions = [
    ('v8t',       0.613, 0.735, 0.526),
    ('v9',        0.677, 0.727, 0.633),
    ('v10',       0.782, 0.768, 0.798),
    ('v11_clean', 0.791, 0.799, 0.783),
    ('v14',       0.8034, 0.811, 0.796),
]
labels, cls_v, pr_v, ws_v = zip(*[(v[0],v[1],v[2],v[3]) for v in versions])
x = np.arange(len(labels))
w = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w, cls_v, w, label='CLS',       color='#2196F3', edgecolor='k', linewidth=0.5)
b2 = ax.bar(x,     pr_v,  w, label='PR-AUC',    color='#4CAF50', edgecolor='k', linewidth=0.5)
b3 = ax.bar(x + w, ws_v,  w, label='WSpearman', color='#FF9800', edgecolor='k', linewidth=0.5)

# Annotate CLS deltas
for i, (c, prev_c) in enumerate(zip(cls_v, [None] + list(cls_v[:-1]))):
    if prev_c is not None:
        delta = c - prev_c
        ax.text(x[i] - w, c + 0.005, f'+{delta:.3f}', ha='center', fontsize=7, color='#1565C0', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0.5, 0.87)
ax.set_ylabel('Score')
ax.set_title('Score Progression Across Model Versions')
ax.legend()
ax.axhline(0.8, color='red', lw=0.8, ls='--', alpha=0.5)
ax.text(len(labels) - 0.5, 0.803, 'CLS=0.80 target', fontsize=8, color='red', ha='right')
# Caption: The v9→v10 jump (+0.105 CLS) from switching to L2 LogisticRegression is the largest single improvement.
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_06_version_comparison.png', dpi=DPI)
plt.close()
print('fig_06 saved')

## Fig 07 — Beta sweep

In [ ]:
# fig_07_beta_sweep.png — line plot of CLS, PR-AUC, WSpearman vs beta (0 to 1).
# Mark optimal beta=0.66. Show behaviour at extremes.

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(beta_vals, cls_vals, color='#2196F3', lw=2,   label='CLS')
ax.plot(beta_vals, pr_vals,  color='#4CAF50', lw=1.5, ls='--', label='PR-AUC')
ax.plot(beta_vals, ws_vals,  color='#FF9800', lw=1.5, ls='--', label='WSpearman')

opt_idx = int(round(best_beta * 100))
ax.axvline(best_beta, color='red', lw=1.2, ls=':', label=f'Optimal beta={best_beta:.2f}')
ax.scatter([best_beta], [cls_vals[opt_idx]], color='red', zorder=5, s=80)
ax.annotate(f'CLS={cls_vals[opt_idx]:.4f}',
            xy=(best_beta, cls_vals[opt_idx]),
            xytext=(best_beta + 0.05, cls_vals[opt_idx] - 0.01),
            fontsize=8, color='red')

ax.text(0.02, cls_vals[0] + 0.002, 'beta=0: pure REG', fontsize=7, color='grey')
ax.text(0.85, cls_vals[-1] + 0.002, 'beta=1: pure CLF', fontsize=7, color='grey')

ax.set_xlabel('Beta (CLF weight in rank blend)')
ax.set_ylabel('Score')
ax.set_title('Beta Sweep: CLF vs REG Blend Weight')
ax.legend()
# Caption: CLS peaks at beta=0.66 (classifier-dominant blend); pure regressor or pure classifier both score ~0.05 lower.
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_07_beta_sweep.png', dpi=DPI)
plt.close()
print('fig_07 saved')

## Fig 08 — OOF predictions vs efficiency

In [ ]:
# fig_08_oof_vs_efficiency.png — scatter: v14 OOF predicted score vs pe_efficiency_pct,
# coloured by family, active/inactive by shape. Annotate MMLV-RT and large ranking errors.

fig, ax = plt.subplots(figsize=(9, 7))
markers = {1: 'o', 0: 'X'}
for fam in FAMILIES:
    for act_val, mk in markers.items():
        sub = df[(df['rt_family'] == fam) & (df['active'] == act_val)].copy()
        if len(sub) == 0: continue
        sub_idx = [df.index.get_loc(i) for i in sub.index]
        ax.scatter(y_eff[sub_idx], oof[sub_idx],
                   color=FAM_COLORS[fam], marker=mk, s=70 if act_val else 45,
                   edgecolors='k', linewidths=0.5, alpha=0.85)

# Annotate MMLV-RT
mmlv_row = df[df['rt_name'] == 'MMLV-RT']
if not mmlv_row.empty:
    pos = df.index.get_loc(mmlv_row.index[0])
    ax.annotate('MMLV-RT', xy=(y_eff[pos], oof[pos]),
                xytext=(y_eff[pos] - 3, oof[pos] - 0.05),
                fontsize=8, arrowprops=dict(arrowstyle='->', lw=0.8))

# Find and annotate largest ranking error among actives
act_idx = np.where(y_bin == 1)[0]
oof_ranks_all = rankdata(oof)
eff_ranks_act = rankdata(y_eff[act_idx])
oof_ranks_act = oof_ranks_all[act_idx]
rank_errors = np.abs(oof_ranks_act - eff_ranks_act)
worst_pos = act_idx[np.argmax(rank_errors)]
ax.annotate(f"Worst: {df.iloc[worst_pos]['rt_name']}",
            xy=(y_eff[worst_pos], oof[worst_pos]),
            xytext=(y_eff[worst_pos] + 1, oof[worst_pos] - 0.07),
            fontsize=7, color='red', arrowprops=dict(arrowstyle='->', color='red', lw=0.8))

ax.set_xlabel('True PE Efficiency (%)')
ax.set_ylabel('v14 OOF Predicted Score')
ax.set_title('OOF Predictions vs True PE Efficiency')
handles = [mpatches.Patch(color=FAM_COLORS[f], label=f) for f in FAMILIES]
handles += [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='grey', ms=8, label='Active'),
            plt.Line2D([0],[0], marker='X', color='w', markerfacecolor='grey', ms=8, label='Inactive')]
ax.legend(handles=handles, fontsize=7, ncol=2)
# Caption: Inactives cluster near pe_efficiency=0 and low predicted scores; mid-range actives show the largest ranking variability.
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_08_oof_vs_efficiency.png', dpi=DPI)
plt.close()
print('fig_08 saved')

## Fig 09 — Ranking quality

In [ ]:
# fig_09_ranking_quality.png — active RTs: true efficiency rank vs predicted OOF rank.
# Point size = efficiency weight. Annotate worst-ranked outlier. Identity line.

act_idx = np.where(y_bin == 1)[0]
act_df3 = df.iloc[act_idx].copy()
act_df3['true_eff_rank']  = rankdata(y_eff[act_idx])
act_df3['pred_oof_rank']  = rankdata(oof[act_idx])
act_df3['weight']         = y_eff[act_idx] + 0.01
act_df3['family']         = families[act_idx]

fig, ax = plt.subplots(figsize=(7, 7))
for fam in FAMILIES:
    sub = act_df3[act_df3['family'] == fam]
    if len(sub) == 0: continue
    ax.scatter(sub['true_eff_rank'], sub['pred_oof_rank'],
               color=FAM_COLORS[fam], s=sub['weight'] * 15 + 30,
               edgecolors='k', linewidths=0.5, alpha=0.85, label=fam)

# Identity line
max_rank = len(act_idx)
ax.plot([0, max_rank + 1], [0, max_rank + 1], 'k--', lw=1, alpha=0.5, label='Perfect ranking')

# Annotate worst outlier
rank_err = np.abs(act_df3['true_eff_rank'] - act_df3['pred_oof_rank'])
worst_row = act_df3.loc[rank_err.idxmax()]
ax.annotate(worst_row['rt_name'],
            xy=(worst_row['true_eff_rank'], worst_row['pred_oof_rank']),
            xytext=(worst_row['true_eff_rank'] - 3, worst_row['pred_oof_rank'] + 1.5),
            fontsize=8, color='red', arrowprops=dict(arrowstyle='->', color='red', lw=0.8))

ax.set_xlabel('True Efficiency Rank (1=lowest)')
ax.set_ylabel('Predicted OOF Rank (1=lowest)')
ax.set_title('Active RT: True Efficiency Rank vs Predicted Rank')
ax.legend(fontsize=7, ncol=2)
# Caption: Points near the identity line indicate correct ranking; point size reflects efficiency weight in WSpearman.
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_09_ranking_quality.png', dpi=DPI)
plt.close()
print('fig_09 saved')

## Fig 10 — Experiment history

In [ ]:
# fig_10_experiment_history.png — horizontal bar chart of all experiments,
# sorted by CLS. Green=improvement, Red=hurt, Grey=neutral. Baseline dashed line.

baseline = 0.7908  # v11_clean
experiments = [
    # (label, cls, outcome)  outcome: 'improve'/'hurt'/'neutral'
    ('v10 (LR classifier)',          0.7824, 'baseline_pre'),
    ('v11 (REG features burst)',      0.7847, 'hurt'),
    ('v11_clean (hbonds_per_res CLF)',0.7908, 'baseline'),
    ('LR C=0.03',                    0.7861, 'hurt'),
    ('LR C=0.05',                    0.7823, 'hurt'),
    ('LR C=0.15',                    0.7887, 'hurt'),
    ('LR C=0.20',                    0.7851, 'hurt'),
    ('PCA n=7',                      0.7277, 'hurt'),
    ('PCA n=10',                     0.7146, 'hurt'),
    ('CLF + avg_log_likelihood',     0.7881, 'neutral'),
    ('CLF + salt_per_res',           0.7643, 'hurt'),
    ('REG + avg_log_likelihood',     0.7813, 'hurt'),
    ('REG + active Ridge',           0.7479, 'hurt'),
    ('CLF + ramachandran_out',       0.7828, 'hurt'),
    ('CLF + aromaticity',            0.7863, 'hurt'),
    ('CLF + n_strands_total',        0.7917, 'improve'),
    ('REG + t70',                    0.7835, 'hurt'),
    ('REG + aspartate_d_charge',     0.7720, 'hurt'),
    ('REG + isoelectric_point',      0.7939, 'improve'),
    ('REG + molecular_weight',       0.7783, 'hurt'),
    ('REG + t70+t80',                0.7767, 'hurt'),
    ('REG + n_hbonds',               0.7818, 'hurt'),
    ('REG + hydrophob_std',          0.7849, 'neutral'),
    ('v14 (n_strands+isoelec+salt)',  0.8034, 'improve'),
]
experiments_sorted = sorted(experiments, key=lambda x: x[1])
labels_e, cls_e, outcomes = zip(*experiments_sorted)

color_map = {'improve': '#4CAF50', 'hurt': '#EF5350',
             'neutral': '#9E9E9E', 'baseline': '#2196F3', 'baseline_pre': '#90CAF9'}
bar_colors = [color_map[o] for o in outcomes]

fig, ax = plt.subplots(figsize=(10, 10))
y_pos = np.arange(len(labels_e))
bars = ax.barh(y_pos, cls_e, color=bar_colors, edgecolor='k', linewidth=0.4)
ax.axvline(baseline, color='navy', lw=1.5, ls='--', label=f'Baseline v11_clean ({baseline:.4f})')
ax.set_yticks(y_pos)
ax.set_yticklabels(labels_e, fontsize=8)
ax.set_xlabel('CLS Score')
ax.set_title('Experiment History: All Tested Configurations')
ax.set_xlim(0.68, 0.82)

# Value labels
for bar, val in zip(bars, cls_e):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=7)

legend_patches = [
    mpatches.Patch(color='#4CAF50', label='Improvement'),
    mpatches.Patch(color='#EF5350', label='Hurt'),
    mpatches.Patch(color='#9E9E9E', label='Neutral'),
    mpatches.Patch(color='#2196F3', label='Baseline (v11_clean)'),
    plt.Line2D([0],[0], color='navy', ls='--', lw=1.5, label='Baseline CLS'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=8)
# Caption: Most feature additions hurt WSpearman; only n_strands_total (CLF), isoelectric_point, and n_salt_bridges (REG) improved CLS.
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_10_experiment_history.png', dpi=DPI)
plt.close()
print('fig_10 saved')

In [ ]:
print('\nAll figures generated:')
for f in sorted(FIG_DIR.glob('*.png')):
    print(f'  {f.name}')
print(f'\nFinal OOF: CLS={cls_final:.4f}  PR-AUC={pr_final:.4f}  WSpearman={ws_final:.4f}')